In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
#
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
from utils.benchmarks import BENCHMARKS_TO_PATHS
from pathlib import Path

In [ ]:
%%time
### cell 0 ###

benchmark_name = "billionaires-statistics-2023"
data = pd.read_csv(Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "Billionaires Statistics Dataset.csv")
factor = 800
data = pd.concat([data] * factor, ignore_index=True)
data.info()

In [ ]:
%%time
### cell 1 ###

data.head(5)

In [ ]:
%%time
### cell 2 ###

data.describe()

In [ ]:
%%time
### cell 3 ###

country_names = data[
    "country"
].value_counts()  # List of how many billionaires there are in the country

In [ ]:
%%time
### cell 4 ###

country_names

In [ ]:
%%time
### cell 5 ###

data_100 = data.loc[:100, ["finalWorth", "category", "country"]]

In [ ]:
%%time
### cell 6 ###

data_100_category = data_100["category"].value_counts()

In [ ]:
%%time
### cell 7 ###

data_usa = data[
    data["country"] == "United States"
]  # We focus on billionaires based in United States

In [ ]:
%%time
### cell 8 ###

data_usa_category = data_usa["category"].value_counts()
data_usa_category

In [ ]:
%%time
### cell 9 ###

data_usa_city = data_usa["city"].value_counts()
print(data_usa_city)
data_usa_city.info()

In [ ]:
%%time
### cell 10 ###

data.head(5)

In [ ]:
%%time
### cell 11 ###

category_list = data["category"].unique().tolist()
category_list

In [ ]:
%%time
### cell 12 ###

data["finalWorth"] = data["finalWorth"].astype(float)

# vectorized groupby + reindex to preserve category_list order
worth_avg = data.groupby("category")["finalWorth"].mean().reindex(category_list)

# build data2 and sort into new_data
data2 = pd.DataFrame(
    {"category_list": worth_avg.index, "worth_average": worth_avg.values}
)
new_data = data2.sort_values("worth_average", ascending=False)

In [ ]:
%%time
### cell 13 ###

data3 = data.dropna()

In [ ]:
%%time
### cell 14 ###

data3.head()

In [ ]:
%%time
### cell 15 ###

data3.info()  # in the new df we have 238 rows for each column

In [ ]:
%%time
### cell 16 ###

data3.countryOfCitizenship.unique()  # and 238 billionaires in usa, others deleted.